# Used Functions and Models

## Models

In [13]:
import torch
from torch import nn

class IndependentHeadsMLP(nn.Module):
    """MLP with a shared body and independent per-label output heads."""

    def __init__(self, input_dim, hidden_dim, n_hidden, n_labels, activation="ReLU"):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)]
        )
        self.heads = nn.ModuleList(
            [nn.Linear(hidden_dim, 1) for _ in range(n_labels)]
        )

        self.activation = nn.ReLU() if activation == "ReLU" else nn.Tanh()

    def forward(self, x):
        x = self.activation(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        return torch.cat([head(x) for head in self.heads], dim=1)  # raw logits

## Functions

In [14]:
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    return avg_loss

def _calculate_val_acc(model, loader):
    global DEVICE

    model.eval()
    running_sum = 0.0
    total_samples = 0.0
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            pred = output.argmax(dim=1)
            correct = (pred == y).sum().item()
            running_sum += correct
            total_samples += y.size(0).item()

    acc = running_sum / total_samples

    return acc

def train_one_epoch(model, train_loader, val_loader, optimizer, criterion):
    global DEVICE

    train_loss = _train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = _calculate_val_acc(model, val_loader)

    return train_loss, val_loss

def objective_func(trial, train_set, val_set,):
    global DEVICE

    EPOCHS = 10
    batch_size = trial.suggest_int('batch_size', 1, 1024)
    lr = trial.suggest_float('lr', 1e-3, 0.5)
    n_hidden = trial.suggest_int('n_hidden', 1, 25)
    hidden_dim = trial.suggest_int('hidden_dim', 1, 512)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    model = IndependentHeadsMLP(input_dim=784, hidden_dim=hidden_dim, n_hidden=n_hidden, n_labels=10).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        train_loss, val_acc = train_one_epoch(model,train_loader ,val_loader, optimizer, criterion)

    writer = SummaryWriter(f'runs/optuna_tensorboard/{trial.number}')

    # Logging
    hparams = {
        "lr": lr,
        "batch_size": batch_size,
        "n_hidden": n_hidden,
        "hidden_dim": hidden_dim
    }

    metrics = {
        "hparam/train_loss": train_loss,
        "hparam/val_acc": val_acc
    }

    writer.add_hparams(hparams, metrics)
    writer.flush()

    return val_acc


# Getting data and runing study

In [15]:
import optuna
# Using MNIST DataSet
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import TensorDataset

mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split( X, y, test_size=5000, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=15000, random_state=42, stratify=y_trainval)

print(f"train size: {len(X_train)}")
print(f"val size: {len(X_val)}")
print(f"test size: {len(X_test)}")

# Data
X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).long()
train_set = TensorDataset(X_train_t, y_train_t)

x_val_t = torch.from_numpy(X_val).float()
y_val_t = torch.from_numpy(y_val).long()
val_set = TensorDataset(x_val_t, y_val_t)


study.optimize(lambda t: objective_func(t, train_set, val_set), n_trials=25, show_progress_bar=True)

train size: 50000
val size: 15000
test size: 5000


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-04-13 13:01:56,391] Trial 6 finished with value: 28.78723404255319 and parameters: {'batch_size': 323, 'lr': 0.4671882607582771, 'n_hidden': 23, 'hidden_dim': 338}. Best is trial 1 with value: 58.206896551724135.
[I 2026-04-13 13:02:28,075] Trial 7 finished with value: 25.19402985074627 and parameters: {'batch_size': 226, 'lr': 0.12408596848720056, 'n_hidden': 12, 'hidden_dim': 146}. Best is trial 1 with value: 58.206896551724135.
[I 2026-04-13 13:02:55,445] Trial 8 finished with value: 28.610169491525422 and parameters: {'batch_size': 255, 'lr': 0.49368731289432016, 'n_hidden': 4, 'hidden_dim': 466}. Best is trial 1 with value: 58.206896551724135.
[I 2026-04-13 13:03:19,111] Trial 9 finished with value: 49.64705882352941 and parameters: {'batch_size': 445, 'lr': 0.23143306778526082, 'n_hidden': 19, 'hidden_dim': 467}. Best is trial 1 with value: 58.206896551724135.
[I 2026-04-13 13:03:27,769] Trial 10 finished with value: 105.5 and parameters: {'batch_size': 969, 'lr': 0.03273

KeyboardInterrupt: 

In [ ]:
# RUN it to see training
# tensorboard --logdir Lab5/runs/optuna_tensorboard